# Markdown增强

目标：从 `content_list` 中找到第一个 `type == "discarded"` 且包含 DOI 的元素，向上下扩展所有相邻的 `discarded` 元素，清洗为一段补充文本，并追加到原始 md 头部，输出到 `md_enhance`。


In [8]:

from __future__ import annotations

import json
import re
import sys
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Iterable


def resolve_repo_root() -> Path:
    candidates = [
        Path('/Data_two/wyw/code/extract4chem_fluorine'),
        Path('/Data/home/ywwang/code/extract4chem_fluorine'),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if (candidate / 'draft').exists():
            return candidate
    raise FileNotFoundError('Cannot resolve repo root.')


def resolve_raw_base(repo_root: Path) -> Path:
    candidates = [
        repo_root / 'data/raw/test-聚酰亚胺文献',
        repo_root / 'data/raw/test-聚酰亚胺文献-2',
        Path('/Data_two/wyw/code/extract4chem_fluorine/data/raw/test-聚酰亚胺文献'),
        Path('/Data_two/wyw/code/extract4chem_fluorine/data/raw/test-聚酰亚胺文献-2'),
        Path('/Data/home/ywwang/code/extract4chem_fluorine/data/raw/test-聚酰亚胺文献'),
        Path('/Data/home/ywwang/code/extract4chem_fluorine/data/raw/test-聚酰亚胺文献-2'),
    ]
    for candidate in candidates:
        if (candidate / 'md_raw').exists() and (candidate / 'content_list').exists():
            return candidate
    raise FileNotFoundError('Cannot resolve raw data base with md_raw and content_list.')


REPO_ROOT = resolve_repo_root()
RAW_BASE = resolve_raw_base(REPO_ROOT)
MD_RAW_DIR = RAW_BASE / 'md_raw'
CONTENT_LIST_DIR = RAW_BASE / 'content_list'
MD_ENHANCE_DIR = RAW_BASE / 'md_enhance'
MD_ENHANCE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(REPO_ROOT / 'draft'))
from pair_content_list_by_similarity import content_item, md_item, pair_items  # noqa: E402

DOI_PATTERN = re.compile(r'(https?://doi\.org/\S+|doi\s*:\s*\S+|10\.\d{4,9}/\S+)', re.IGNORECASE)
CONTROL_PATTERN = re.compile(r'[\x00-\x1f\x7f-\x9f]+')
SPACE_PATTERN = re.compile(r'\s+')


def load_content_items(path: Path) -> list[dict]:
    data = json.loads(path.read_text(encoding='utf-8'))
    if isinstance(data, list):
        return data
    if isinstance(data, dict):
        for key in ('content_list', 'items', 'data'):
            value = data.get(key)
            if isinstance(value, list):
                return value
    raise TypeError(f'Unsupported content_list format: {path}')


def contains_doi(text: str) -> bool:
    return bool(DOI_PATTERN.search(text or ''))


def normalize_text(text: str) -> str:
    cleaned = CONTROL_PATTERN.sub(' ', text or '')
    cleaned = SPACE_PATTERN.sub(' ', cleaned).strip()
    return cleaned


def find_first_doi_discarded_cluster(items: list[dict]) -> tuple[int, int, list[dict]] | None:
    signals = ["discarded", "footer","header"]
    for idx, item in enumerate(items):
        if item.get('type') not in signals:
            continue
        if not contains_doi(item.get('text', '')):
            continue
        start = idx
        while start - 1 >= 0 and items[start - 1].get('type') in signals:
            start -= 1
        end = idx
        while end + 1 < len(items) and items[end + 1].get('type') in signals:
            end += 1
        return start, end, items[start:end + 1]
    
    # 降级
    for idx, item in enumerate(items):
        if item.get('type') not in signals:
            continue
        start = idx
        while start - 1 >= 0 and items[start - 1].get('type') in signals:
            start -= 1
        end = idx
        while end + 1 < len(items) and items[end + 1].get('type') in signals:
            end += 1
        return start, end, items[start:end + 1]
    
    return None


def clean_cluster_lines(cluster: Iterable[dict]) -> list[str]:
    lines: list[str] = []
    seen: set[str] = set()
    for item in cluster:
        line = normalize_text(item.get('text', ''))
        if not line:
            continue
        if not re.search(r'[A-Za-z0-9\u4e00-\u9fff]', line):
            continue
        key = line.casefold()
        if key in seen:
            continue
        seen.add(key)
        lines.append(line)
    return lines


def build_enhanced_md(original_md: str, discarded_lines: list[str]) -> str:
    if not discarded_lines:
        return original_md
    header = '[MINERU_DISCARDED_META]\n' + '\n'.join(discarded_lines) + '\n[/MINERU_DISCARDED_META]'
    body = original_md.lstrip('\ufeff').lstrip()
    return header + '\n\n' + body


@dataclass
class EnhanceResult:
    file_name: str
    content_list_name: str | None
    match_type: str
    match_score: float | None
    matched_cluster: bool
    cluster_start: int | None
    cluster_end: int | None
    line_count: int
    output_path: str


In [9]:

null_count = 0
md_items = [md_item(path) for path in sorted(MD_RAW_DIR.glob('*.md'))]
content_items = [content_item(path) for path in sorted(CONTENT_LIST_DIR.glob('*.json'))]
pair_rows = pair_items(md_items, content_items, min_score=0.85)
pair_map = {
    row['md_name']: row
    for row in pair_rows
    if row['md_name'] is not None
}

results: list[EnhanceResult] = []

for md_path in sorted(MD_RAW_DIR.glob('*.md')):
    pair_info = pair_map.get(md_path.name, {
        'content_name': None,
        'match_type': 'unmatched',
        'score': None,
    })
    content_name = pair_info['content_name']
    content_path = None if content_name is None else CONTENT_LIST_DIR / content_name
    output_path = MD_ENHANCE_DIR / md_path.name

    if content_path is None or not content_path.exists():
        output_path.write_text(md_path.read_text(encoding='utf-8'), encoding='utf-8')
        results.append(
            EnhanceResult(
                file_name=md_path.name,
                content_list_name=content_name,
                match_type=pair_info['match_type'],
                match_score=pair_info['score'],
                matched_cluster=False,
                cluster_start=None,
                cluster_end=None,
                line_count=0,
                output_path=str(output_path),
            )
        )
        continue

    items = load_content_items(content_path)
    cluster_info = find_first_doi_discarded_cluster(items)
    if cluster_info is None:
        null_count += 0
        discarded_lines: list[str] = []
        cluster_start = None
        cluster_end = None
        matched_cluster = False
    else:
        cluster_start, cluster_end, cluster = cluster_info
        discarded_lines = clean_cluster_lines(cluster)
        matched_cluster = True

    enhanced_md = build_enhanced_md(md_path.read_text(encoding='utf-8'), discarded_lines)
    output_path.write_text(enhanced_md, encoding='utf-8')
    results.append(
        EnhanceResult(
            file_name=md_path.name,
            content_list_name=content_name,
            match_type=pair_info['match_type'],
            match_score=pair_info['score'],
            matched_cluster=matched_cluster,
            cluster_start=cluster_start,
            cluster_end=cluster_end,
            line_count=len(discarded_lines),
            output_path=str(output_path),
        )
    )

for item in results:
    print(asdict(item))


{'file_name': '3D Printing of Thermal Insulating PolyimideCellulose Nanocrystal Composite Aerogels with Low Dimensional Shrin.md', 'content_list_name': '3D Printing of Thermal Insulating PolyimideCellulose Nanocrystal Composite Aerogels with Low Dimensional Shrin_content_list.json', 'match_type': 'exact_normalized', 'match_score': 1.0, 'matched_cluster': True, 'cluster_start': 25, 'cluster_end': 29, 'line_count': 4, 'output_path': '/Data_two/wyw/code/extract4chem_fluorine/data/raw/test-聚酰亚胺文献/md_enhance/3D Printing of Thermal Insulating PolyimideCellulose Nanocrystal Composite Aerogels with Low Dimensional Shrin.md'}
{'file_name': 'A Novel Application of Electroactive Polyimide Doped with Gold Nanoparticles As a Chemiresistor Sensor for Hyd.md', 'content_list_name': 'A Novel Application of Electroactive Polyimide Doped with Gold Nanoparticles As a Chemiresistor Sensor for Hyd_content_list.json', 'match_type': 'exact_normalized', 'match_score': 1.0, 'matched_cluster': True, 'cluster_sta

In [10]:
output_path

PosixPath('/Data_two/wyw/code/extract4chem_fluorine/data/raw/test-聚酰亚胺文献/md_enhance/Polyimide-Derived Supramolecular Systems Containing Various Amounts of Azochromophore for Optical Storage Uses.md')

In [11]:
md_names = {md.stem for md in MD_RAW_DIR.glob('*.md')}
json_names = {json_file.stem for json_file in CONTENT_LIST_DIR.glob('*.json')}


In [12]:
enhanced_md

'[MINERU_DISCARDED_META]\npolymers\nMDPI\nPolymers 2023, 15, 1056. https://doi.org/10.3390/polym15041056\nhttps://www.mdpi.com/journal/polymers\n[/MINERU_DISCARDED_META]\n\nArticle\n\n# Polyimide-Derived Supramolecular Systems Containing Various Amounts of Azochromophore for Optical Storage Uses\n\nAndreea Irina Barzic  $^{1}$ , Ion Sava  $^{1}$ , Raluca Marinica Albu  $^{1}$ , Cristian Ursu  $^{1}$ , Gabriela Lisa  $^{2}$  and Iuliana Stoica  $^{1, *}$\n\n$^{1}$  "Petru Poni" Institute of Macromolecular Chemistry, 700487 Iasi, Romania  \n2 Faculty of Chemical Engineering and Environmental Protection "Cristofor Simionescu", "Gheorghe Asachi" Technical University of Iasi-Romania, 700050 Iasi, Romania  \n* Correspondence:stoica_iuliana@icmpp.ro\n\nAbstract: The progress of digital technologies demands more speed and larger storage capacity. Optical storage systems have the advantage of being cheap, fast and capacious. This article explores the potential use of polyimide-based films as a 